In [ ]:
import os
import sys
module_path = os.path.abspath(os.path.join('../src'))
if module_path not in sys.path:
    sys.path.append(module_path)

import vsopy.util
import vsopy.phot

OBJ_NAME='SZ_Lyn'
SESSION_TAG='2026/20260123'
IMAGE_ROOT = '/srv/public/img'
WORK_ROOT = '/srv/public'

session = vsopy.util.Session(tag=SESSION_TAG, name=OBJ_NAME)
layout = vsopy.util.WorkLayout(WORK_ROOT)
session_layout = layout.get_session(session)
settings = vsopy.util.Settings(session_layout.settings_file_path)

primary = 'V'
secondary = ['B', 'Rc', 'Ic']

In [ ]:
provider = vsopy.phot.BatchDataProvider(session_layout)
provider.images_.rename_column('filter', 'band')
provider.measured_.rename_column('M', 'm')

In [ ]:
from astropy.table import join
import numpy as np
batches = join(provider.batches_["batch_id", "airmass", "time"],
               provider.batch_images_["batch_id", "image_id"],
               keys=["batch_id"])
batches = batches[batches["batch_id"] <= 80]
instr = join(batches["batch_id", "image_id", "airmass", "time"],
            join(provider.images_["image_id", "band"],
                provider.measured_["image_id", "auid", "m"],
                keys=["image_id"]),
            keys=["image_id"])
#instr = instr[(instr["auid"] != "000-BBP-190") & (instr["auid"] != "000-BBP-198")]
target = instr[instr["auid"] == provider.target_auid]

In [ ]:
from astropy.table import QTable
def get_std_ci(sequence, band):
    pair = join(sequence[sequence["band"] == band[0]],
                sequence[sequence["band"] == band[1]],
                keys=["auid"],
                table_names=["A", "B"])
    result = QTable([pair["auid"]])
    result["CI"] = pair["M_A"]["mag"] - pair["M_B"]["mag"]
    result["CI_err"] = np.sqrt(pair["M_A"]["err"] **2 + pair["M_B"]["err"]**2)
    return result

def get_instr_ci(instr, band):
    pair = join(instr[instr["band"] == band[0]]["batch_id", "auid", "m"],
                instr[instr["band"] == band[1]]["batch_id", "auid", "m"],
                keys=["batch_id", "auid"],
                table_names=["A", "B"])
    result = join(pair["batch_id", "auid"],
                  instr[instr["band"] == band[0]]["batch_id", "auid", "airmass"],
                  keys=["batch_id", "auid"])
    result["ci"] = pair["m_A"]["mag"] - pair["m_B"]["mag"]
    result["ci_err"] = np.sqrt(pair["m_A"]["err"] **2 + pair["m_B"]["err"]**2)
    return result

def get_instr_dm(instr, sequence, band):
    mM = join(instr[instr["band"] == band]["batch_id", "auid", "m"],
         sequence[sequence["band"] == band]["auid", "M"],
         keys=["auid"])
    result = join(mM["batch_id", "auid"],
                  instr[instr["band"] == band]["batch_id", "auid", "airmass"],
                  keys=["batch_id", "auid"])
    result["dm"] = mM["m"]["mag"] - mM["M"]["mag"]
    result["dm_err"] = np.sqrt(mM["m"]["err"]**2 + mM["M"]["err"]**2)
    return result
#get_std_ci(provider.sequence_, band=['V', 'B'])
#get_instr_ci(target, band=['V', 'B'])
get_instr_dm(instr, provider.sequence_, band='V')

In [ ]:
from uncertainties import unumpy

def colmpose_X(dataCI):
    return np.vstack([np.ones(len(dataCI)),
            dataCI["airmass"],
            dataCI["CI"].value,
            dataCI["airmass"] * dataCI["CI"].value,
            dataCI["CI"].value * dataCI["CI"].value,
            # np.exp(dataCI["CI"].value)
            ]).T

def solve_mult(X, y):
    Q = X.T @ X
    QI = np.linalg.inv(Q)
    XX = QI @ X.T
    a = XX @ y
    S = (X @ a - y)
    s2 = np.sum(S**2) / (len(y) - len(a))
    sigma = np.diag(QI) * s2
    return unumpy.uarray(a, np.sqrt(sigma)), S

def solve_ci(instr, std):
    dataCI = join(instr, std, keys=["auid"])
    X=colmpose_X(dataCI)
    return solve_mult(X, dataCI["ci"].value)

def solve_dm(instr, std):
    datadM = join(instr, std, keys=["auid"])
    X=colmpose_X(datadM)
    return solve_mult(X, datadM["dm"].value)

instrB = get_instr_ci(instr, band=['V', 'B'])
instrR = get_instr_ci(instr, band=['V', 'Rc'])
instrI = get_instr_ci(instr, band=['V', 'Ic'])
instrV = get_instr_dm(instr, provider.sequence_, band='V')
stdB = get_std_ci(provider.sequence_, band=['V', 'B'])
stdR = get_std_ci(provider.sequence_, band=['V', 'Rc'])
stdI = get_std_ci(provider.sequence_, band=['V', 'Ic'])

aV, S= solve_dm(instrV, stdB)
bV, SV = solve_ci(instrB, stdB)
bR, SR = solve_ci(instrR, stdR)
bI, SI = solve_ci(instrI, stdI)
#solve_ci(get_instr_ci(instr, band=['V', 'B']), get_std_ci(provider.sequence_, band=['V', 'B']))
#solve_dm(get_instr_dm(instr, provider.sequence_, band='V'), get_std_ci(provider.sequence_, band=['V', 'B']))
aV, bV, bR, bI

In [ ]:
instr_res = instrV["batch_id", "auid"]
instr_res["r"] = S
instr_res["r2"] = S**2
g = instr_res["auid", "r2"].group_by("auid")
per_auid = g.groups.aggregate(np.sum)
per_auid["r2"] = np.sqrt(per_auid["r2"] / len(instr_res))
per_auid.sort("r2")
per_auid

In [ ]:
import matplotlib.pyplot as plt
from scipy import stats

def plot_res(S):
    S2 = S**2
    s2 = np.sum(S2) / (len(S2) - len(aV))
    cen = S2 < 3 * s2
    S_cen = S[cen]
    fig = plt.figure(figsize=(10.24,10.24))
    ax = plt.subplot(221)
    ax.hist(S, bins=100)
    ax = plt.subplot(222)
    ax.hist(S_cen, bins=100)
    ax = plt.subplot(223)
    stats.probplot(S, plot=ax)
    ax = plt.subplot(224)
    stats.probplot(S_cen, plot=ax)
    plt.show()

plot_res(S)
plot_res(SV)
plot_res(SR)
plot_res(SI)

In [ ]:
import itertools


#batched = [solve_dm(chunk, stdB) for chunk in chunker(instrV, 7)]

BATCH_SIZE=5
batched = [solve_dm(instrV[(instrV["batch_id"] >= i) & (instrV["batch_id"] < i+BATCH_SIZE)], stdB)
           for i in np.unique(instrV["batch_id"])]
a, _ = zip(*batched)

fig = plt.figure(figsize=(10.24,10.24))
ax = plt.subplot()
ax.plot([x[0].nominal_value for x in a], "o")
plt.show()


In [ ]:

for auid in per_auid["auid"]:
    S = instr_res[instr_res["auid"] == auid]["r"]
    plt.title(auid)
    ax = plt.subplot(121)
    ax.hist(S, bins=100)
    ax = plt.subplot(122)
    stats.probplot(S, plot=ax)
    plt.tight_layout()
    plt.show()
